# Introduction to the Ray AI Libraries: An example of using Ray data, Ray Train, Ray Tune, Ray Serve to implement a XGBoost regression model

© 2025, Anyscale. All Rights Reserved

💻 **Launch Locally**: You can run this notebook locally, but performance will be reduced.

🚀 **Launch on Cloud**: A Ray Cluster with 4 GPUs (Click [here](http://console.anyscale.com/register) to easily start a Ray cluster on Anyscale) is recommended to run this notebook.

Let's start with a quick end-to-end example to get a sense of what the Ray AI Libraries can do.
<div class="alert alert-block alert-info">
<b> Here is the roadmap for this notebook:</b>
<ul>
    <li>Overview of the Ray AI Libraries</li>
    <li>Quick end-to-end example</li>
    <ul>
      <li>Vanilla XGBoost code</li>
      <li>Hyperparameter tuning with Ray Tune</li>
      <li>Distributed training with Ray Train</li>
      <li>Serving an ensemble model with Ray Serve</li>
      <li>Batch inference with Ray Data</li>
    </ul>
</ul>
</div>

**Imports**

In [ ]:
# (Optional): If you get an XGBoostError at import, you might have to `brew install libomp` before importing xgboost again
!brew install libomp

In [1]:
import asyncio
import fastapi
import pandas as pd
import requests
# macos: If you get an XGBoostError at import, you might have to `brew install libomp` before importing xgboost again
import xgboost
from pydantic import BaseModel
from sklearn.model_selection import train_test_split

import ray
import ray.tune
import ray.train
from ray.train.xgboost import XGBoostTrainer as RayTrainXGBoostTrainer
from ray.train import RunConfig
import ray.data
import ray.serve

2026-06-29 22:48:42,759	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-06-29 22:48:42,970	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-06-29 22:48:43,038	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## 1. Overview of the Ray AI Libraries

<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Ray_AI_Libraries/Ray+AI+Libraries.png" width="700px" loading="lazy">

Built on top of Ray Core, the Ray AI Libraries inherit all the performance and scalability benefits offered by Core while providing a convenient abstraction layer for machine learning. These Python-first native libraries allow ML practitioners to distribute individual workloads, end-to-end applications, and build custom use cases in a unified framework.

The Ray AI Libraries bring together an ever-growing ecosystem of integrations with popular machine learning frameworks to create a common interface for development.

|<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Introduction_to_Ray_AIR/e2e_air.png" width="100%" loading="lazy">|
|:-:|
|Ray AI Libraries enable end-to-end ML development and provides multiple options for integrating with other tools and libraries from the MLOps ecosystem.|



## 2. Quick end-to-end example

For this classification task, you will apply a simple [XGBoost](https://xgboost.readthedocs.io/en/stable/) (a gradient boosted trees framework) model to the June 2021 [New York City Taxi & Limousine Commission's Trip Record Data](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page). 

The full dataset contains millions of samples of yellow cab rides, and the goal is to predict the tip amount.

**Dataset features**
* **`passenger_count`**
    * Float (whole number) representing number of passengers.
* **`trip_distance`** 
    * Float representing trip distance in miles.
* **`fare_amount`**
    * Float representing total price including tax, tip, fees, etc.
* **`tolls_amount`**
    * Float representing the total paid on tolls if any.

**Target**
* **`trip_amount`**
    * Float representing the total paid as tips

### 2.1 Vanilla XGboost code

Let's start with the vanilla XGBoost code to predict the tip amount for a NYC taxi cab data.

In [2]:
features = [
    "passenger_count", 
    "trip_distance",
    "fare_amount",
    "tolls_amount",
]

label_column = "tip_amount"

Define a function to load the data and split into train and test

In [ ]:
def load_data():
    base_dir = "./data/"
    path = f"{base_dir}yellow_tripdata_2021-06.parquet"
    df = pd.read_parquet(path, columns=features + [label_column])
    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df[label_column], test_size=0.2, random_state=42
    )
    dtrain = xgboost.DMatrix(X_train, label=y_train)
    dtest = xgboost.DMatrix(X_test, label=y_test)
    return dtrain, dtest

Define a function to run `xgboost.train` given some hyperparameter dictionary `params`

In [ ]:
storage_folder = "./storage" # Modify this path to your local folder if it runs on your local environment

In [5]:
from pathlib import Path
model_path = Path(storage_folder) / "model.ubj"

def my_xgboost_func(params):    
    evals_result = {}
    dtrain, dtest = load_data()
    bst = xgboost.train(
        params, 
        dtrain, 
        num_boost_round=10, 
        evals=[(dtest, "eval")], 
        evals_result=evals_result,
    )
    # Use Path
    bst.save_model(model_path)
    print(f"{evals_result['eval']}")
    return {"eval-rmse": evals_result["eval"]["rmse"][-1]}

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "max_depth": 6,
    "eta": 0.1,
}
my_xgboost_func(params)

[0]	eval-rmse:2.60843
[1]	eval-rmse:2.54281
[2]	eval-rmse:2.48831
[3]	eval-rmse:2.44303
[4]	eval-rmse:2.40449
[5]	eval-rmse:2.37260
[6]	eval-rmse:2.34634
[7]	eval-rmse:2.32456
[8]	eval-rmse:2.30673
[9]	eval-rmse:2.29203
OrderedDict([('rmse', [2.608426869346233, 2.542814037050064, 2.4883053362509235, 2.4430317102680115, 2.404488045053629, 2.372601212036056, 2.3463407935552407, 2.3245557417100606, 2.3067344778194263, 2.292026706979947])])


{'eval-rmse': 2.292026706979947}

### 2.2 Hyperparameter tuning with Ray Tune

Let's use Ray Tune to run distributed hyperparameter tuning for the XGBoost model.

In [6]:
import os
from ray.tune import CheckpointConfig, RunConfig
# 1. Force the engine flag at the top of the block
os.environ["RAY_AIR_NEW_OUTPUT"] = "0"

tuner = ray.tune.Tuner(  # Create a tuner
    my_xgboost_func,  # Pass it the training function which Ray Tune calls Trainable.
    param_space={  # Pass it the parameter space to search over
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",
        "max_depth": 6,
        "eta": ray.tune.uniform(0.01, 0.3),
    },
    run_config=RunConfig(
        storage_path=storage_folder,
        checkpoint_config=CheckpointConfig(
            checkpoint_at_end=False, checkpoint_frequency=0
        ),),
    tune_config=ray.tune.TuneConfig(  # Tell it which metric to tune
        metric="eval-rmse",
        mode="min",
        num_samples=10,
    ),
    # verbose = 0,
)

results = tuner.fit()  # Run the tuning job
print(results.get_best_result().config)  # Get back the best hyperparameters

2026-06-29 22:49:00,791	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2026-06-29 22:49:01,492	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `Tuner(...)`.


(my_xgboost_func pid=19260) [0]	eval-rmse:2.66258
(my_xgboost_func pid=19265) [0]	eval-rmse:2.64522
(my_xgboost_func pid=19267) [0]	eval-rmse:2.62174
(my_xgboost_func pid=19264) [0]	eval-rmse:2.52834
(my_xgboost_func pid=19263) [0]	eval-rmse:2.53437
(my_xgboost_func pid=19262) [0]	eval-rmse:2.52453
(my_xgboost_func pid=19258) [0]	eval-rmse:2.56236
(my_xgboost_func pid=19259) [0]	eval-rmse:2.59408
(my_xgboost_func pid=19260) [1]	eval-rmse:2.63971
(my_xgboost_func pid=19266) [0]	eval-rmse:2.66855
(my_xgboost_func pid=19261) [0]	eval-rmse:2.50536
(my_xgboost_func pid=19265) [1]	eval-rmse:2.60736
(my_xgboost_func pid=19263) [1]	eval-rmse:2.43110
(my_xgboost_func pid=19259) [1]	eval-rmse:2.51935
(my_xgboost_func pid=19267) [1]	eval-rmse:2.56551
(my_xgboost_func pid=19262) [1]	eval-rmse:2.41811
(my_xgboost_func pid=19258) [1]	eval-rmse:2.47067
(my_xgboost_func pid=19260) [2]	eval-rmse:2.61800
(my_xgboost_func pid=19266) [1]	eval-rmse:2.65113
(my_xgboost_func pid=19264) [1]	eval-rmse:2.42309


Trial name,checkpoint_dir_name,date,done,eval-rmse,experiment_tag,hostname,iterations_since_restore,node_ip,pid,time_since_restore,time_this_iter_s,time_total_s,timestamp,training_iteration,trial_id
my_xgboost_func_6539d_00000,,2026-06-29_22-49-09,True,2.23925,0_eta=0.1629,BLVLTM057.local,1,127.0.0.1,19258,4.53901,4.53901,4.53901,1782798549,1,6539d_00000
my_xgboost_func_6539d_00001,,2026-06-29_22-49-09,True,2.40145,1_eta=0.0520,BLVLTM057.local,1,127.0.0.1,19265,4.54108,4.54108,4.54108,1782798549,1,6539d_00001
my_xgboost_func_6539d_00002,,2026-06-29_22-49-09,True,2.22745,2_eta=0.2029,BLVLTM057.local,1,127.0.0.1,19263,4.41965,4.41965,4.41965,1782798549,1,6539d_00002
my_xgboost_func_6539d_00003,,2026-06-29_22-49-09,True,2.26863,3_eta=0.1192,BLVLTM057.local,1,127.0.0.1,19259,4.54507,4.54507,4.54507,1782798549,1,6539d_00003
my_xgboost_func_6539d_00004,,2026-06-29_22-49-09,True,2.22578,4_eta=0.2173,BLVLTM057.local,1,127.0.0.1,19262,4.41577,4.41577,4.41577,1782798549,1,6539d_00004
my_xgboost_func_6539d_00005,,2026-06-29_22-49-09,True,2.49387,5_eta=0.0300,BLVLTM057.local,1,127.0.0.1,19260,4.50592,4.50592,4.50592,1782798549,1,6539d_00005
my_xgboost_func_6539d_00006,,2026-06-29_22-49-09,True,2.53356,6_eta=0.0225,BLVLTM057.local,1,127.0.0.1,19266,4.58588,4.58588,4.58588,1782798549,1,6539d_00006
my_xgboost_func_6539d_00007,,2026-06-29_22-49-09,True,2.32119,7_eta=0.0824,BLVLTM057.local,1,127.0.0.1,19267,4.55707,4.55707,4.55707,1782798549,1,6539d_00007
my_xgboost_func_6539d_00008,,2026-06-29_22-49-09,True,2.22244,8_eta=0.2461,BLVLTM057.local,1,127.0.0.1,19261,4.49563,4.49563,4.49563,1782798549,1,6539d_00008
my_xgboost_func_6539d_00009,,2026-06-29_22-49-09,True,2.22625,9_eta=0.2117,BLVLTM057.local,1,127.0.0.1,19264,4.53868,4.53868,4.53868,1782798549,1,6539d_00009


(my_xgboost_func pid=19258) [8]	eval-rmse:2.24666
(my_xgboost_func pid=19263) [9]	eval-rmse:2.22745
(my_xgboost_func pid=19263) OrderedDict([('rmse', [2.5343689306536024, 2.431096245663961, 2.360266812369164, 2.313773113126263, 2.282845428974792, 2.2621341157657207, 2.247799667218475, 2.2381104550982496, 2.2315308899727198, 2.2274473999256372])])
(my_xgboost_func pid=19267) [8]	eval-rmse:2.33794
(my_xgboost_func pid=19266) [8]	eval-rmse:2.54631
(my_xgboost_func pid=19262) [9]	eval-rmse:2.22578
(my_xgboost_func pid=19264) [8]	eval-rmse:2.22984
(my_xgboost_func pid=19261) [9]	eval-rmse:2.22244
(my_xgboost_func pid=19260) [9]	eval-rmse:2.49387
(my_xgboost_func pid=19258) [9]	eval-rmse:2.23925
(my_xgboost_func pid=19265) [9]	eval-rmse:2.40145
(my_xgboost_func pid=19259) [9]	eval-rmse:2.26863
(my_xgboost_func pid=19267) [9]	eval-rmse:2.32119
(my_xgboost_func pid=19266) [9]	eval-rmse:2.53356
(my_xgboost_func pid=19264) [9]	eval-rmse:2.22625


2026-06-29 22:49:09,832	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/Users/alwin.babu/Documents/RayAnyscale/Introduction_Notebook/storage/my_xgboost_func_2026-06-29_22-48-57' in 0.0228s.
2026-06-29 22:49:09,842	INFO tune.py:1039 -- Total run time: 8.35 seconds (8.30 seconds for the tuning loop).


{'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'max_depth': 6, 'eta': 0.24607873823111848}


Here is a diagram that shows what Tune does:

It is effectively scheduling many trials and returning the best performing one.

<img src="https://bair.berkeley.edu/static/blog/tune/tune-arch-simple.png" width="700px" loading="lazy">

### 2.3. Distributed training with Ray Train

In case your training data is too large, your training might take a long time to complete.

To speed it up, shard the dataset across training workers and perform distributed XGBoost training.

Let's redefine `load_data` to now load a different slice of the data given the worker index/rank.

In [ ]:
def load_data():
    # find out which training worker is running this code
    train_ctx = ray.train.get_context()
    worker_rank = train_ctx.get_world_rank()
    print(f"Loading data for worker {worker_rank}...")

    # build path based on training worker rank
    month = (worker_rank + 1) % 12
    year = 2021 + (worker_rank + 1) // 12
    
    base_dir = (
        "./data/"
    )
    path = f"{base_dir}yellow_tripdata_{year}-{month:02}.parquet"

    # same as before
    df = pd.read_parquet(path, columns=features + [label_column])
    X_train, X_test, y_train, y_test = train_test_split(
        df[features], df[label_column], test_size=0.2, random_state=42
    )
    dtrain = xgboost.DMatrix(X_train, label=y_train)
    dtest = xgboost.DMatrix(X_test, label=y_test)
    return dtrain, dtest

Now we can run distributed XGBoost training using Ray Train's XGBoostTrainer - similar trainers exist for other popular ML frameworks.

In [8]:
trainer = RayTrainXGBoostTrainer(  # Create a trainer
    my_xgboost_func,  # Pass it the training function
    scaling_config=ray.train.ScalingConfig(
        num_workers=2, use_gpu=False
    ),  # Define how many training workers
    train_loop_config=params,  # Pass it the hyperparameters
)

trainer.fit()  # Run the training job

(TrainController pid=19341) Requesting resources: {'CPU': 1} * 2
(TrainController pid=19341) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2
(TrainController pid=19341) Started training worker group of size 2: 
(TrainController pid=19341) - (ip=127.0.0.1, pid=19354) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=19341) - (ip=127.0.0.1, pid=19355) world_rank=1, local_rank=1, node_rank=0
(RayTrainWorker pid=19354) [22:49:34] Task [xgboost.ray-rank=00000000]:543fd012e6f9ae1c281c6ea201000000 got rank 0
(RayTrainWorker pid=19354) Loading data for worker 0...


(RayTrainWorker pid=19354) Loading data for worker 0...


(TrainController pid=19341) [22:49:35] [0]	eval-rmse:2.28346
(TrainController pid=19341) [22:49:35] [1]	eval-rmse:2.25069
(TrainController pid=19341) [22:49:35] [2]	eval-rmse:2.22460
(TrainController pid=19341) [22:49:35] [3]	eval-rmse:2.20430
(TrainController pid=19341) [22:49:35] [4]	eval-rmse:2.18836
(TrainController pid=19341) [22:49:35] [5]	eval-rmse:2.17259
(TrainController pid=19341) [22:49:35] [6]	eval-rmse:2.15961
(TrainController pid=19341) [22:49:35] [7]	eval-rmse:2.14910
(TrainController pid=19341) [22:49:35] [8]	eval-rmse:2.14038
(TrainController pid=19341) [22:49:35] [9]	eval-rmse:2.13346
(RayTrainWorker pid=19354) OrderedDict([('rmse', [np.float64(2.283455977036048), np.float64(2.2506874095991365), np.float64(2.2245956490092267), np.float64(2.204302370137811), np.float64(2.1883569504768654), np.float64(2.1725859682897077), np.float64(2.1596106275006597), np.float64(2.149095348540642), np.float64(2.140382007941743), np.float64(2.133460611893429)])])


(RayTrainWorker pid=19354) OrderedDict([('rmse', [np.float64(2.283455977036048), np.float64(2.2506874095991365), np.float64(2.2245956490092267), np.float64(2.204302370137811), np.float64(2.1883569504768654), np.float64(2.1725859682897077), np.float64(2.1596106275006597), np.float64(2.149095348540642), np.float64(2.140382007941743), np.float64(2.133460611893429)])])


Result(metrics=None, checkpoint=None, error=None, path='/Users/alwin.babu/ray_results/ray_train_run-2026-06-29_22-49-29', metrics_dataframe=None, best_checkpoints=[], _storage_filesystem=<pyarrow._fs.LocalFileSystem object at 0x30e19a1f0>, return_value={'eval-rmse': np.float64(2.133460611893429)})

Here is a diagram that shows what Train does:

A train controller will create training workers and execute the training function on each worker.

Ray Train delegates the distributed training to the underlying XGBoost framework.

<img src="https://docs.ray.io/en/latest/_images/overview.png" width="700px" loading="lazy">

### 2.4 Serving an ensemble model with Ray Serve

Ray Serve allows for distributed serving of models and complex inference pipelines.

Here is a diagram showing how to deploy an ensemble model with Ray Serve:

<img src="https://images.ctfassets.net/xjan103pcp94/3DJ7vVRxYIvcFO7JmIUMCx/77a45caa275ffa46f5135f4d6726dd4f/Figure_2_-_Fanout_and_ensemble.png" width="700px" loading="lazy">

Here is how the resulting code looks like:

In [11]:
app = fastapi.FastAPI()

class Payload(BaseModel):
    passenger_count: int
    trip_distance: float
    fare_amount: float
    tolls_amount: float


@ray.serve.deployment
@ray.serve.ingress(app)
class Ensemble:
    def __init__(self, model1, model2):
        self.model1 = model1
        self.model2 = model2

    @app.post("/predict")
    async def predict(self, data: Payload) -> dict:
        model1_prediction, model2_prediction = await asyncio.gather(
            self.model1.predict.remote([data.model_dump()]),
            self.model2.predict.remote([data.model_dump()]),
        )
        # --- THE FIX ---
        # Extract the first prediction scalar element [0] from each returned array
        p1 = float(model1_prediction[0])
        p2 = float(model2_prediction[0])

        out = {"prediction": (p1 + p2) / 2}
        return out


@ray.serve.deployment
class Model:
    def __init__(self, path: str):
        self._model = xgboost.Booster()
        self._model.load_model(path)

    def predict(self, data: list[dict]) -> list[float]:
        # Make prediction
        dmatrix = xgboost.DMatrix(pd.DataFrame(data))
        model_prediction = self._model.predict(dmatrix)
        return model_prediction


# Run the deployment
handle = ray.serve.run(
    Ensemble.bind(
        model1=Model.bind(model_path),
        model2=Model.bind(model_path),
    ),
    route_prefix="/ensemble"
)

WARNING 2026-06-29 22:56:56,146 serve 19217 -- There are multiple deployments with the same name 'Model'. Renaming one to 'Model_1'.
INFO 2026-06-29 22:56:56,152 serve 19217 -- Connecting to existing Serve app in namespace "serve". New http_options/controller_options will not be applied.
(ServeController pid=19246) INFO 2026-06-29 22:56:56,234 controller 19246 -- Deploying new version of Deployment(name='Model', app='default') (initial target replicas: 1).
(ServeController pid=19246) INFO 2026-06-29 22:56:56,235 controller 19246 -- Deploying new version of Deployment(name='Model_1', app='default') (initial target replicas: 1).
(ServeController pid=19246) INFO 2026-06-29 22:56:56,236 controller 19246 -- Deploying new version of Deployment(name='Ensemble', app='default') (initial target replicas: 1).
(ServeController pid=19246) INFO 2026-06-29 22:56:56,342 controller 19246 -- Stopping 1 replicas of Deployment(name='Model', app='default') with outdated versions.
(ServeController pid=19246

Let's make an HTTP request to the Ray Serve instance.

In [12]:
requests.post(
    "http://localhost:8000/ensemble/predict",
    json={  # Use json parameter instead of params
        "passenger_count": 1,
        "trip_distance": 2.5,
        "fare_amount": 10.0,
        "tolls_amount": 0.5,
    },
).json()

(ServeReplica:default:Model pid=19670) /Users/alwin.babu/miniforge3/envs/ray_project/lib/python3.11/site-packages/ray/serve/_private/replica.py:3349: UserWarning: Calling sync method 'predict' directly on the asyncio loop. In a future version, sync methods will be run in a threadpool by default. Ensure your sync methods are thread safe or keep the existing behavior by making them `async def`. Opt into the new behavior by setting RAY_SERVE_RUN_SYNC_IN_THREADPOOL=1.
(ServeReplica:default:Model pid=19670)   warnings.warn(
(ServeReplica:default:Ensemble pid=19669) INFO 2026-06-29 22:57:04,489 default_Ensemble e7to7j7c 5c1cd83d-43c7-4710-b3ad-caf15b56b759 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x127098710>.
(ServeReplica:default:Model pid=19670) INFO 2026-06-29 22:57:04,508 default_Model np45f35s 5c1cd83d-43c7-4710-b3ad-caf15b56b759 -- CALL predict OK 7.0ms


{'prediction': 2.0076115131378174}

(ServeReplica:default:Model_1 pid=19668) INFO 2026-06-29 22:57:04,511 default_Model_1 j2xtbhqs 5c1cd83d-43c7-4710-b3ad-caf15b56b759 -- CALL predict OK 4.8ms
(ServeReplica:default:Ensemble pid=19669) INFO 2026-06-29 22:57:04,512 default_Ensemble e7to7j7c 5c1cd83d-43c7-4710-b3ad-caf15b56b759 -- POST /ensemble 200 38.2ms


### 2.5 Batch inference with Ray Data

Ray Data allows for distributed data processing through streaming execution across a heterogeneous cluster of CPUs and GPUs.

This makes Ray Data ideal for workloads like compute-intensive data processing, data ingestion, and batch inference.

In [ ]:
class OfflinePredictor:
    def __init__(self):
        # Load expensive state
        self._model = xgboost.Booster()
        self._model.load_model(model_path)

    def predict(self, data: list[dict]) -> list[float]:
        # Make prediction in batch
        dmatrix = xgboost.DMatrix(pd.DataFrame(data))
        model_prediction = self._model.predict(dmatrix)
        return model_prediction

    def __call__(self, batch: dict) -> dict:
        batch["predictions"] = self.predict(batch)
        return batch


# Apply the predictor to the validation dataset
prediction_pipeline = (
    ray.data.read_parquet(
        "./data/yellow_tripdata_2021-03.parquet"
    )
    .select_columns(features)
    .map_batches(OfflinePredictor, concurrency=(2, 10))
)

2026-06-29 22:58:34,074	WARNING tqdm_ray.py:319 -- tqdm is not installed. Progress bars will be disabled.
2026-06-29 22:58:35,517	INFO parquet_datasource.py:1810 -- Estimated parquet encoding ratio is 9.752.
2026-06-29 22:58:35,517	INFO parquet_datasource.py:1876 -- Estimated parquet reader batch size at 883012 rows
2026-06-29 22:58:35,957	WARNING util.py:647 -- The argument ``concurrency`` is deprecated in Ray 2.51. Please specify argument ``compute`` instead. For more information, see https://docs.ray.io/en/master/data/transforming-data.html#stateful-transforms.


After defining the pipeline, we can execute it in a distributed manner by writing the output to a sink

In [15]:
prediction_pipeline.write_parquet("./xgboost_predictions") #update this to your local path if runs on your local

2026-06-29 22:58:49,422	INFO streaming_executor.py:193 -- Starting execution of Dataset dataset_4_0. Full logs are in /tmp/ray/session_2026-06-29_22-48-57_571332_19217/logs/ray-data
2026-06-29 22:58:49,423	INFO streaming_executor.py:194 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> ActorPoolMapOperator[MapBatches(OfflinePredictor)] -> TaskPoolMapOperator[Write]
[2026-06-29 22:58:49,438 E 19217 21213123] core_worker.cc:2149: Actor with class name: 'MapWorker(MapBatches(OfflinePredictor))' and ID: '298afb7ffdf98aba6b3bbf0d01000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details.
2026-06-29 22:58:49,441	INFO __init__.py:56 -- Progress will be logged because stdout is a non-interactive terminal.
2026-06-29 22:58:49,449	WARNING resource_manager.py:766 

Let's inspect the produced predictions.

In [18]:
!ls xgboost_predictions/

2_1832b3cdeefe40619416d2dff727a796_000000-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000001-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000002-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000003-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000004-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000005-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000006-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000007-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000008-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000009-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000010-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000011-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000012-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000013-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000014-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000015-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000016-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000017-0.parquet
2_1832b3cdeefe40619416d2dff727a796_000018-0.parquet
2_1832b3cdee

### 2.6 Clean up

In [ ]:
# Run this cell for file cleanup 
!rm -rf {storage_folder}/xgboost_predictions/
!rm {model_path}